<h2>Which features should I keep?

and:

Which features does my model actually find useful?

Feature selection basically involves deciding what models to keep or remove

Feature importance involves determining how useful the model found each feature

<h2>Random Forest gives us feature importance

This is one of the reasons Random Forest is useful.

In [2]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

#model.fit(X_train, y_train)

In [3]:
#TO determine how useful RF found each model, we can say

In [4]:
#model.feature_importances_

You might get:

[0.35, 0.25, 0.20, 0.12, 0.08]

In [5]:
#But looking at those values, we don't know which feature has what value, so we can say

In [7]:
"""importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance)"""

'importance = pd.DataFrame({\n    "feature": X_train.columns,\n    "importance": model.feature_importances_\n})\n\nimportance = importance.sort_values(\n    "importance",\n    ascending=False\n)\n\nprint(importance)'

<h3>THOSE NUMBERS DO NOT REPRESENT PERCENTAGE

It doesn't mean:

"Distance causes 35% of the coverage problem."

And it doesn't mean:

"If distance increases by 1 km, coverage gets 35% worse."

It means that, according to this particular Random Forest importance measure, distance contributed substantially to the model's decision-making relative to the other features.

<h5>Sometimes, feature importance can be misleading. Especially between closely related features.

<h3>Consequently, another way is using permuation importance

The basic idea is wonderfully simple.

Suppose our trained model performs:

Accuracy = 92%

Then we randomly shuffle one feature.
|
For example:

distance

Suppose:

Original accuracy = 92%

After scrambling the distance values,
Accuracy = 74%

That's a big drop.

That suggests the model relied heavily on distance.

Now shuffle:

population_density

and we get:

92% → 90%

Smaller drop.

That suggests population density was less important to the model in that context.

The general idea is:

Important feature
→ shuffle it
→ performance drops a lot

Less important feature
→ shuffle it
→ performance changes little

In [9]:
#To test it out

In [10]:
from sklearn.inspection import permutation_importance

#Feature importance is determined using the test data

In [11]:
"""result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)"""

'result = permutation_importance(\n    model,\n    X_test,\n    y_test,\n    n_repeats=10,\n    random_state=42\n)'

In [12]:
#Then
"""importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance)"""

'importance = pd.DataFrame({\n    "feature": X_test.columns,\n    "importance": result.importances_mean\n})\n\nimportance = importance.sort_values(\n    "importance",\n    ascending=False\n)\n\nprint(importance)'

<h3>Another perspective is provided by Logistic Regression. Coefficients

Suppose we have:

support_calls
satisfaction
monthly_charge

and our fitted Logistic Regression gives coefficients:

support_calls     +1.8
satisfaction      -2.1
monthly_charge    +0.4

The signs tell us the direction of the relationship in the model:

Positive coefficient
→ higher feature value is associated with higher model score for class 1

Negative coefficient
→ higher feature value is associated with lower model score for class 1

Suppose:

support_calls → +2.0
satisfaction  → -2.5

You might say:

"Higher support calls are associated with a higher tendency toward the churn class, while higher satisfaction is associated with a lower tendency toward churn, according to the fitted model."

In [13]:
data = {
    "distance_km": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0,
                    5.5, 6.0, 6.5, 7.0, 7.5, 8.0],

    "elevation_diff_m": [10, 20, 15, 30, 25, 40, 35, 50,
                         45, 60, 55, 70, 65, 80, 75, 90],

    "population_density": [500, 800, 1000, 1500, 1800, 2200, 2500, 3000,
                           3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000],

    "terrain_score": [1, 1, 2, 1, 2, 2, 3, 2,
                      3, 3, 2, 3, 3, 4, 4, 4],

    "land_cover_score": [1, 1, 1, 2, 2, 2, 3, 2,
                         3, 3, 3, 4, 4, 4, 5, 5],

    "coverage_gap": [0, 0, 0, 0, 0, 0, 0, 0,
                     0, 1, 1, 1, 1, 1, 1, 1]
}

In [14]:
import pandas as pd

df = pd.DataFrame(data)
df.head()

,distance_km,elevation_diff_m,population_density,terrain_score,land_cover_score,coverage_gap
0,0.5,10,500,1,1,0
1,1.0,20,800,1,1,0
2,1.5,15,1000,2,1,0
3,2.0,30,1500,1,2,0
4,2.5,25,1800,2,2,0


In [16]:
X  = df[["distance_km", "elevation_diff_m", "population_density", "terrain_score", "land_cover_score"]]
y  = df["coverage_gap"]


In [17]:
from sklearn.model_selection import train_test_split

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [21]:
from sklearn.ensemble import RandomForestClassifier

In [22]:
model  = RandomForestClassifier(
    n_estimators  = 10,
    max_depth = 5
)

In [23]:
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, n_estimators=10)

In [26]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

In [30]:
predictions = model.predict(X_test)
print(predictions)

[0 0 0 1]


In [31]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0


In [32]:
#Now to see the feature importances
#no need to import feature importance, RF has that normally

In [33]:
print(model.feature_importances_)

[0.475 0.4   0.1   0.    0.025]


In [34]:
#Which  values belongs to which?

In [35]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance)

              feature  importance
0         distance_km       0.475
1    elevation_diff_m       0.400
2  population_density       0.100
4    land_cover_score       0.025
3       terrain_score       0.000


In [36]:
#TRying out permutation, 
from sklearn.inspection import permutation_importance

In [37]:
result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

print(result)

{'importances_mean': array([0., 0., 0., 0., 0.]), 'importances_std': array([0., 0., 0., 0., 0.]), 'importances': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])}


In [38]:
permutation_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

print(permutation_df)

              feature  importance
0         distance_km         0.0
1    elevation_diff_m         0.0
2  population_density         0.0
3       terrain_score         0.0
4    land_cover_score         0.0


<h3>This also teaches us something important about the two methods

Random Forest feature_importances_

It asks roughly:

"How much did this feature contribute to the tree-splitting decisions?"

Permutation importance

It asks:

"How much does my model's performance suffer when I destroy the information in this feature?"

In [40]:
print(result.importances_mean)
print(result.importances_std)

[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
